In [2]:
import geopandas as gpd
import folium

# === 1. Load parking zones ===
zones = gpd.read_file("bird_parking.geojson")

# === 2. Load OD flows (Bird) ===
od = gpd.read_file("od_bird.geojson")

# === 3. Ensure same CRS ===
zones = zones.to_crs(epsg=4326)
od = od.to_crs(epsg=4326)

# === 4. Create base map (Torino center) ===
m = folium.Map(location=[45.07, 7.69], zoom_start=12)

m

In [8]:
od.columns


Index(['ZONASTAT_orig', 'ZONASTAT_dest', 'trips', 'ZONASTAT_x', 'orig_lat',
       'orig_lon', 'ZONASTAT_y', 'dest_lat', 'dest_lon', 'distance_m',
       'time_min', 'monetary_cost', 'time_cost', 'GC_scooter', 'geometry'],
      dtype='str')

In [3]:
import branca.colormap as cm

# === create colormap ===
colormap = cm.linear.YlOrRd_09.scale(
    zones["parking_minutes"].min(),
    zones["parking_minutes"].max()
)

colormap.caption = "Average Parking Duration (minutes)"
colormap.add_to(m)

# === add zones layer ===
folium.GeoJson(
    zones,
    style_function=lambda feature: {
        "fillColor": colormap(
            feature["properties"]["parking_minutes"]
        ) if feature["properties"]["parking_minutes"] is not None else "transparent",
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["parking_minutes"],
        aliases=["Parking duration:"]
    )
).add_to(m)

m

In [4]:
# === add OD flows ===
folium.GeoJson(
    od,
    style_function=lambda feature: {
        "color": "blue",
        "weight": min(feature["properties"]["trips"] / 500, 5),
        "opacity": 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ZONASTAT_orig", "ZONASTAT_dest", "trips"],
        aliases=["From:", "To:", "Trips:"]
    )
).add_to(m)

m

In [6]:
import pandas as pd

In [9]:
import geopandas as gpd
import folium
import branca.colormap as cm

# parking
zones = gpd.read_file("bird_parking.geojson")

# OD
od_morning = gpd.read_file("od_bird_morning.geojson")
od_evening = gpd.read_file("od_bird_evening.geojson")

# CRS
zones = zones.to_crs(epsg=4326)
od_morning = od_morning.to_crs(epsg=4326)
od_evening = od_evening.to_crs(epsg=4326)

# fix nulls
zones["parking_minutes"] = zones["parking_minutes"].fillna(0)

In [11]:
# === base map ===
m_morning = folium.Map(location=[45.07, 7.69], zoom_start=12)

# === colormap ===
colormap_m = cm.linear.YlOrRd_09.scale(
    zones["parking_minutes"].min(),
    zones["parking_minutes"].max()
)
colormap_m.caption = "Parking Duration (Morning)"
colormap_m.add_to(m_morning)

# === parking layer ===
folium.GeoJson(
    zones,
    style_function=lambda feature: {
        "fillColor": colormap_m(feature["properties"]["parking_minutes"]),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7
    }
).add_to(m_morning)

# === OD morning ===
folium.GeoJson(
    od_morning,
    style_function=lambda feature: {
        "color": "blue",
        "weight": min(feature["properties"]["trips"] / 50, 5),
        "opacity": 0.7
    }
).add_to(m_morning)

m_morning

In [14]:
m_morning.save("overlay_morning_bird.html")

In [15]:
# === base map ===
m_evening = folium.Map(location=[45.07, 7.69], zoom_start=12)

# === colormap ===
colormap_e = cm.linear.YlOrRd_09.scale(
    zones["parking_minutes"].min(),
    zones["parking_minutes"].max()
)
colormap_e.caption = "Parking Duration (Evening)"
colormap_e.add_to(m_evening)

# === parking layer ===
folium.GeoJson(
    zones,
    style_function=lambda feature: {
        "fillColor": colormap_e(feature["properties"]["parking_minutes"]),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7
    }
).add_to(m_evening)

# === OD evening ===
folium.GeoJson(
    od_evening,
    style_function=lambda feature: {
        "color": "green",
        "weight": min(feature["properties"]["trips"] / 50, 5),
        "opacity": 0.7
    }
).add_to(m_evening)

m_evening

In [16]:
m_evening.save("overlay_evening_bird.html")